# exp98 — contacts-v1 train-set rollouts: interactive explorer

**1,000,000 rollouts** (1000 training targets × 1000 each) from the tuned contacts-v1 1.5B model (eval loss 2.7566), for [Open-Athena/MarinFold#98](https://github.com/Open-Athena/MarinFold/issues/98).

Each rollout is a sampled contacts-v1 contact set, scored (precision/recall/F1, all sep≥6) vs the document's ground-truth contacts. We keep every rollout's metrics + model NLL + predicted contacts, and save the best-recall & best-F1 rollout per target verbatim.

**No login needed** — the data is in the *public* `open-athena/MarinFold` HF bucket. Run top to bottom.


In [ ]:
!pip -q install -U 'huggingface_hub>=1.5' pandas pyarrow matplotlib ipywidgets  # bucket fs needs hf_hub>=1.5

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from huggingface_hub import HfFileSystem

# Anonymous (token=False) read of the public bucket — no auth.
fs = HfFileSystem(token=False)
BASE = "buckets/open-athena/MarinFold/data/contacts-v1-train-rollouts-exp98"
def rp(name, **kw):
    with fs.open(f'{BASE}/{name}', 'rb') as f:
        return pd.read_parquet(f, **kw)

## Per-target summary (1000 targets)


In [ ]:
t = rp('per_target_summary.parquet')
print(len(t), 'targets')
t[['entry_id','L','n_gt','best_recall','best_f1','mean_rec','mean_gen_tokens','tokens_per_s']].describe().round(3)

## Overview — best-of-1000 accuracy (all sep≥6)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, col in zip(ax, ['best_recall','best_f1']):
    a.hist(t[col], bins=30, color='steelblue', edgecolor='white')
    a.axvline(t[col].mean(), color='crimson', ls='--', label=f'mean {t[col].mean():.2f}')
    a.set_xlabel(col + ' (best of 1000)'); a.set_ylabel('# targets'); a.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(t.L, t.best_f1, s=8, alpha=.4, label='best F1')
ax[0].scatter(t.L, t.best_recall, s=8, alpha=.4, label='best recall')
ax[0].set_xlabel('L'); ax[0].set_ylabel('best of 1000'); ax[0].legend(); ax[0].set_title('accuracy vs length')
sc = ax[1].scatter(t.n_gt, t.best_f1, c=t.L, s=8, alpha=.5, cmap='viridis')
fig.colorbar(sc, ax=ax[1], label='L'); ax[1].set_xlabel('# GT contacts'); ax[1].set_ylabel('best F1'); ax[1].set_title('best F1 vs contact count')
plt.tight_layout(); plt.show()

## Load the full per-rollout table + ground truth
`rollout_metrics_all.parquet` (1M rows) has every rollout's per-band metrics, model `nll`/`nll_per_tok`, and `pred` (flattened predicted contact pairs). `targets.parquet` has the ground-truth contacts.


In [ ]:
ALL = rp('rollout_metrics_all.parquet')
TG  = rp('targets.parquet')
GT  = {r.entry_id: {tuple(sorted(map(int, p))) for p in r.gt_contacts} for r in TG.itertuples()}
groups = {e: g for e, g in ALL.groupby('entry_id')}   # per-target slices (one pass)
best = rp('best_rollouts.parquet')
print('rollouts:', len(ALL), '| targets:', len(groups))

## Drill into one target


In [ ]:
def explore(entry):
    m = groups[entry]; b = best[best.entry_id == entry].iloc[0]
    print(f'{entry}  L={int(b.L)}  n_gt={int(b.n_gt)}  ({len(m)} rollouts)')
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].scatter(m.all_rec, m.all_prec, s=6, alpha=.25)
    ax[0].set_xlabel('recall'); ax[0].set_ylabel('precision'); ax[0].set_title('per-rollout (all sep>=6)')
    ax[1].hist(m.all_f1, bins=30, color='steelblue', edgecolor='white')
    ax[1].set_xlabel('F1'); ax[1].set_ylabel('# rollouts'); ax[1].set_title('F1 distribution')
    plt.tight_layout(); plt.show()
    print(f'best F1 {b.best_f1_f1:.3f} (rec {b.best_f1_recall:.3f})  |  best recall {b.best_recall_recall:.3f}')
    print('--- best-F1 rollout document (truncated) ---'); print(b.best_f1_document[:500], '...')

TOP = t.sort_values('best_f1').iloc[-1].entry_id
explore(TOP)

## Does model confidence (NLL) predict accuracy?
Within each target, correlate the rollout's length-normalized NLL with its F1 (negative = more-confident rollouts are more accurate), and compare single-rollout selection rules.


In [ ]:
def spearman(x, y):
    x = pd.Series(x).rank().to_numpy(); y = pd.Series(y).rank().to_numpy()
    return float(np.corrcoef(x, y)[0, 1]) if x.std() and y.std() else np.nan
rows = []
for e, g in groups.items():
    g = g.dropna(subset=['nll_per_tok','all_f1'])
    if len(g) < 10: continue
    i = int(g.nll_per_tok.to_numpy().argmin())
    rows.append((spearman(g.nll_per_tok, g.all_f1), g.all_f1.mean(), g.all_f1.max(), g.all_f1.to_numpy()[i]))
nll = pd.DataFrame(rows, columns=['rho','mean','oracle','min_nll'])
print(f"within-target Spearman(NLL/tok, F1): mean {nll.rho.mean():+.3f}; negative in {100*(nll.rho<0).mean():.0f}% of targets")
print(f"F1: random {nll['mean'].mean():.3f} | min-NLL (label-free) {nll.min_nll.mean():.3f} | oracle {nll.oracle.mean():.3f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(nll.rho.dropna(), bins=30, color='steelblue', edgecolor='white'); ax[0].axvline(nll.rho.mean(), color='crimson', ls='--')
ax[0].set_xlabel('per-target Spearman(NLL/tok, F1)'); ax[0].set_title('confidence tracks accuracy (neg = yes)')
ax[1].bar(['random','min-NLL','oracle'], [nll['mean'].mean(), nll.min_nll.mean(), nll.oracle.mean()], color=['#bbb','#6aa','#088'])
ax[1].set_ylabel('F1'); ax[1].set_title('single-rollout selection'); plt.tight_layout(); plt.show()

## Consensus: vote across rollouts vs one rollout
Score each candidate pair by how often it is emitted across the 1000 rollouts (optionally weighted by `softmax(-NLL/token)`), keep the **top-K** (K = typical rollout size — label-free), and score that set. Voting beats any single rollout.


In [ ]:
def pairset(flat):
    f = list(map(int, flat)); return {(f[i], f[i+1]) for i in range(0, len(f), 2)}
def prf(pred, gt):
    if not pred: return 0., 0., 0.
    tp = len(pred & gt); p = tp/len(pred); r = tp/len(gt) if gt else float('nan')
    return p, r, (2*p*r/(p+r) if p+r > 0 else 0.)
def vote_f1(g, gt, weighted=False, T=0.3):
    preds = [pairset(x) for x in g['pred']]; sizes = np.array([len(p) for p in preds])
    if weighted:
        n = g.nll_per_tok.to_numpy(); w = np.exp(-(n - np.nanmin(n))/T); w = w/w.sum()
    else:
        w = np.ones(len(preds))/len(preds)
    sc = {}
    for wi, p in zip(w, preds):
        for pr in p: sc[pr] = sc.get(pr, 0.) + wi
    K = max(1, round(sizes.mean()))
    top = {pr for _, pr in sorted(((s, pr) for pr, s in sc.items()), reverse=True)[:K]}
    return prf(top, gt)[2]

g = groups[TOP]; gt = GT[TOP]
print(f'{TOP}: vote top-K F1 {vote_f1(g, gt):.3f} | wvote {vote_f1(g, gt, True):.3f} | best single {g.all_f1.max():.3f} | mean single {g.all_f1.mean():.3f}')

# aggregate over a 100-target sample (the committed CSV covers all 1000)
rng = np.random.default_rng(0)
samp = rng.choice(list(groups), size=100, replace=False)
res = [(groups[e].all_f1.mean(), groups[e].all_f1.max(), vote_f1(groups[e], GT[e]), vote_f1(groups[e], GT[e], True)) for e in samp]
R = pd.DataFrame(res, columns=['single_mean','single_oracle','vote_topK','wvote_topK'])
print('mean F1 over 100 targets:', {k: round(v,3) for k,v in R.mean().items()})
R.mean().plot.bar(color=['#bbb','#088','#26a','#a4c']); plt.ylabel('F1'); plt.title('consensus vs single (100-target sample)'); plt.tight_layout(); plt.show()

## Interactive picker


In [ ]:
try:
    import ipywidgets as W
    order = t.sort_values('best_f1', ascending=False)
    opts = [(f'{r.entry_id}  (L={int(r.L)}, bestF1={r.best_f1:.2f})', r.entry_id) for r in order.itertuples()]
    W.interact(lambda entry: explore(entry), entry=W.Dropdown(options=opts, description='target'))
except Exception as e:
    print('ipywidgets unavailable — call explore("AF-...") directly:', e)